# MNIST Raw-LiRA Saturation Sweep v3 (K to 512)

**v3 changes vs v2 (2026-07-05):**
- `K_LADDER` extended to 512 (v2's K=256 verdict was STILL CLIMBING, +0.082 nats on the last doubling).
- Disjoint per-seed member/non-member draws (`EVAL_LIMIT` 512 -> 2560): pooled Wilson counts are now over distinct examples.
- `censored_below_detection_floor` flag: a conservative 0 with a positive point/GDP estimate means *below the estimator's detection floor*, never "attack recovers 0%".
- Plateau verdict computed on valid, non-censored cells only.
- Pre-flight sanity cell (aborts on failure) since the bundle has no tests/.

Upload `saturation_bundle_v3.zip` when prompted.

In [ ]:
from google.colab import files
up = files.upload()  # choose saturation_bundle_v3.zip
import zipfile
zname = [f for f in up if f.endswith(".zip")][0]
zipfile.ZipFile(zname).extractall(".")
print("Extracted", zname)

In [ ]:
%pip -q install opacus dp-accounting pyyaml scipy
import torch
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU ONLY -- ABORT: runtime will be ~10-20x slower; Runtime > Change runtime type > T4 GPU")

In [ ]:
# --- PRE-FLIGHT SANITY CHECKS (abort before any training) -------------------
# Embedded from research/validate_estimator_scorer_20260705.py (the bundle has
# no tests/). Verifies the estimator core + corrected scorer logic and that the
# bundle actually contains the 2026-07-05 fixes. Aborts the notebook if any fail.
import sys, random, statistics
sys.path.insert(0, "src"); sys.path.insert(0, "codex")
from dp_audit_tightness.privacy.empirical import estimate_empirical_lower_bound
import run_raw_lira_pilot as pilot
import inspect

random.seed(608)
kw = dict(delta=1e-5, align_event_to_score_direction=True,
          require_member_favoring=True, report_confidence_supported_lower_bound=True)
def est(m, n, direction="higher"):
    return estimate_empirical_lower_bound(member_scores=m, nonmember_scores=n,
                                          score_direction=direction, **kw).epsilon_lower_empirical
checks = []
m=[random.gauss(0,1) for _ in range(640)]; n=[random.gauss(0,1) for _ in range(640)]
checks.append(("T1 no-signal -> eps=0", est(m,n) < 0.05))
m=[random.gauss(3,1) for _ in range(640)]; n=[random.gauss(0,1) for _ in range(640)]
checks.append(("T2a signal+higher -> eps>0", est(m,n,"higher") > 0.5))
checks.append(("T2b signal+lower -> eps=0", est(m,n,"lower") < 0.05))
K,B=32,640
mem_a,non_a,mem_s,non_s=[],[],[],[]
for _ in range(B):
    sl=[random.gauss(2.0,0.5) for _ in range(K)]; t=random.gauss(2.0,0.5)
    mem_a.append(statistics.fmean(sl[K//2:])-statistics.fmean(sl[:K//2]))
    mem_s.append(statistics.fmean(sl[K//2:])-t)
for _ in range(B):
    sl=[random.gauss(2.0,0.5) for _ in range(K)]; t=random.gauss(2.0,0.5)
    non_a.append(statistics.fmean(sl)-t); non_s.append(statistics.fmean(sl)-t)
checks.append(("T3a buggy asymmetric scorer, no signal -> spurious eps",
               max(est(mem_a,non_a,"higher"), est(mem_a,non_a,"lower")) > 0.2))
checks.append(("T3b fixed symmetric scorer, no signal -> eps~0", est(mem_s,non_s,"higher") < 0.05))
mem2=[statistics.fmean([random.gauss(2.0,0.5) for _ in range(K//2)])-random.gauss(0.8,0.4) for _ in range(B)]
non2=[statistics.fmean([random.gauss(2.0,0.5) for _ in range(K)])-random.gauss(2.0,0.5) for _ in range(B)]
e4=est(mem2,non2,"higher")
checks.append(("T4 symmetric scorer + real signal -> eps>0", e4 > 0.5))
checks.append(("T5 validity-gate comparison well-defined", (e4 > 0.771) in (True, False)))
# bundle fix markers
src = inspect.getsource(pilot.raw_lira_scores)
checks.append(("bundle has SYMMETRIC scorer", "fmean(out_losses) - float(target_" in src.replace("\n","")))
checks.append(("bundle has disjoint sampler", hasattr(pilot, "sample_query_indices_disjoint")))
checks.append(("bundle has quality flags", hasattr(pilot, "apply_quality_flags")))
failed = [name for name, ok in checks if not ok]
for name, ok in checks: print(("PASS " if ok else "FAIL ") + name)
if failed:
    raise SystemExit(f"SANITY CHECKS FAILED: {failed} -- do NOT run the experiment.")
print("\nAll sanity checks passed -- safe to run.")

In [ ]:
!python codex/run_mnist_saturation.py

In [ ]:
import json, pandas as pd
rows = json.load(open("codex/results/mnist_saturation/mnist_saturation_summary.json"))
df = pd.DataFrame([r for r in rows if r.get("attack") == "passive_raw_lira"])
cols = ["k_shadows","epsilon_lower_conservative","epsilon_lower_point","epsilon_lower_gdp",
        "epsilon_upper_tighter","tightness_ratio_tighter","validity","censored"]
display(df[cols].sort_values("k_shadows"))
verdict = json.load(open("codex/results/mnist_saturation/mnist_saturation_verdict.json"))
print("VERDICT:", verdict["verdict"])
print("last delta:", verdict["last_delta_vs_prev_k"])

In [ ]:
import matplotlib.pyplot as plt
ok = df[(df.validity == "ok") & (df.censored == "not_censored")]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].semilogx(ok.k_shadows, ok.epsilon_lower_conservative, "o-", base=2, label="Wilson conservative")
ax[0].semilogx(ok.k_shadows, ok.epsilon_lower_point, "s--", base=2, label="point estimate")
ax[0].axhline(ok.epsilon_upper_tighter.iloc[0], color="r", ls=":", label="eps_upper (PLD)")
ax[0].set_xlabel("K shadows"); ax[0].set_ylabel("eps_lower"); ax[0].legend(); ax[0].set_title("Saturation ladder (valid, non-censored)")
ax[1].semilogx(ok.k_shadows, ok.tightness_ratio_tighter, "o-", base=2)
ax[1].set_xlabel("K shadows"); ax[1].set_ylabel("tightness vs PLD"); ax[1].set_title("Tightness")
plt.tight_layout(); plt.show()

In [ ]:
!cd codex/results && zip -r ../../mnist_saturation_results_v3.zip mnist_saturation
from google.colab import files
files.download("mnist_saturation_results_v3.zip")